# Лабораторная работа 1. Методы решения задач линейной алгебры


## Общий функционал


In [1]:
import math
import numpy as np

np.set_printoptions(precision=10, suppress=False, linewidth=140)
PIVOT_TOLERANCE = 1e-12
DENOMINATOR_TOLERANCE = 1e-12


In [2]:
def fmt(value):
    return f"{value:.12g}" if isinstance(value, (float, np.floating)) else str(value)


def print_table(headers, rows):
    rows = [[fmt(value) for value in row] for row in rows]
    widths = [max(len(str(header)), *(len(row[i]) for row in rows)) for i, header in enumerate(headers)]
    def line(row):
        return " | ".join(str(value).rjust(width) for value, width in zip(row, widths))
    print(line(headers))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(line(row))


def print_matrix(title, matrix):
    print(f"\n{title}:\n{np.asarray(matrix)}")


def max_abs_difference(first, second):
    return float(np.max(np.abs(np.asarray(first) - np.asarray(second))))


## Задание 1.1. LU-разложение

### Условие

Реализовать алгоритм $LU$-разложения матриц (с выбором главного элемента)
в виде программы. Используя разработанное программное обеспечение,
решить систему линейных алгебраических уравнений (СЛАУ).
Для матрицы СЛАУ вычислить определитель и обратную матрицу. 

### Требования лаборанта

Обязательно реализовать выбор опорного элемента.

Необходимо в результате работы программы вывести:
1. Матрицы $L$ и $U$.
2. Результат умножения $L · U$.
3. Решение системы $Ax = b$.
4. Обратную матрицу $A⁻¹$, найденную через $LU$-разложение.
5. Определитель матрицы $A$, найденный через $LU$-разложение.
6. Проверку $A · A⁻¹ = E$.
7. Проверку $L · U = P · A$, где $P$ — матрица перестановок.


### Входные данные

$\begin{cases} 9 \cdot x_1 - 5 \cdot x_2 - 6 \cdot x_3 + 3 \cdot x_4 = -8 \\ x_1 - 7 \cdot x_2 + x_3 = 38 \\ 3 \cdot x_1 - 4 \cdot x_2 + 9 \cdot x_3 = 47 \\ 6 \cdot x_1 - x_2 + 9 \cdot x_3 + 8 \cdot x_4 = -8 \end{cases}$


In [3]:
data1 = {'A': [[9, -5, -6, 3], [1, -7, 1, 0], [3, -4, 9, 0], [6, -1, 9, 8]], 'b': [-8, 38, 47, -8]}


### Реализация

LU-разложение с выбором ведущего элемента, прямая и обратная подстановки (§1.1.1).


In [4]:
def lu(A):
    """Разложить A на матрицы P, L и U, чтобы P · A = L · U

    P — матрица перестановок строк
    L — нижняя треугольная матрица
    U — верхняя треугольная матрица
    Выбор главного элемента помогает избежать деления на слишком маленькое число
    """
    n = len(A)
    U = A.copy().astype(float)
    L = np.eye(n)
    P = np.eye(n)
    swaps = 0

    for k in range(n):
        # Ищем в текущем столбце элемент с наибольшим модулем ниже диагонали
        # Его строка станет ведущей (опорной) на этом шаге
        row = k + np.argmax(np.abs(U[k:, k]))
        pivot_row_scale = float(np.max(np.abs(U[row, k:])))
        pivot_tolerance = PIVOT_TOLERANCE * pivot_row_scale
        if abs(U[row, k]) <= pivot_tolerance:
            raise ValueError("Матрица вырождена или численно плохо обусловлена: не найден надёжный опорный элемент.")

        if row != k:
            # Меняем строки местами в U и P
            U[[k, row]] = U[[row, k]]
            P[[k, row]] = P[[row, k]]
            # Уже найденные элементы L в этих строках тоже должны поменяться
            if k:
                L[[k, row], :k] = L[[row, k], :k]
            swaps += 1

        # Обнуляем элементы под диагональю текущего столбца
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]

    return L, U, P, swaps


def solve_lu(L, U, P, b):
    """Решить систему A · x = b, используя уже найденные L, U и P

    Сначала прямой подстановкой решаем L · y = P · b,
    затем обратной подстановкой решаем U · x = y
    """
    n = len(b)
    y = np.zeros(n)
    x = np.zeros(n)
    pb = P @ b

    # Прямая подстановка: вычисляем y слева направо
    for i in range(n):
        y[i] = (pb[i] - L[i, :i] @ y[:i]) / L[i, i]

    # Обратная подстановка: вычисляем x справа налево
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]

    return x


def main(data):
    A = np.asarray(data["A"], dtype=float)
    b = np.asarray(data["b"], dtype=float)
    L, U, P, swaps = lu(A)
    x = solve_lu(L, U, P, b)
    E = np.eye(len(A))
    inverse = np.column_stack([solve_lu(L, U, P, E[:, j]) for j in range(len(A))])
    determinant = (-1) ** swaps * np.prod(np.diag(U))
    for name, matrix in (("A", A), ("L", L), ("U", U), ("P", P),
                         ("L @ U", L @ U), ("P @ A", P @ A),
                         ("Обратная матрица через LU", inverse), ("A @ A_inverse", A @ inverse)):
        print_matrix(name, matrix)
    print("\nРешение x:", x)
    print(f"Определитель через LU: {determinant:.12g}; перестановок строк: {swaps}")
    print_table(["Проверка", "Максимальное расхождение"], [
        ("L @ U = P @ A", max_abs_difference(L @ U, P @ A)),
        ("A @ A_inverse = E", max_abs_difference(A @ inverse, E)),
        ("A @ x = b", max_abs_difference(A @ x, b)),
    ])
    return {"L": L, "U": U, "P": P, "x": x, "inverse": inverse, "determinant": determinant}

result1 = main(data1)



A:
[[ 9. -5. -6.  3.]
 [ 1. -7.  1.  0.]
 [ 3. -4.  9.  0.]
 [ 6. -1.  9.  8.]]

L:
[[ 1.            0.            0.            0.          ]
 [ 0.1111111111  1.            0.            0.          ]
 [ 0.6666666667 -0.3620689655  1.            0.          ]
 [ 0.3333333333  0.3620689655  0.7642585551  1.          ]]

U:
[[ 9.           -5.           -6.            3.          ]
 [ 0.           -6.4444444444  1.6666666667 -0.3333333333]
 [ 0.            0.           13.6034482759  5.8793103448]
 [ 0.            0.            0.           -5.3726235741]]

P:
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]

L @ U:
[[ 9. -5. -6.  3.]
 [ 1. -7.  1.  0.]
 [ 6. -1.  9.  8.]
 [ 3. -4.  9.  0.]]

P @ A:
[[ 9. -5. -6.  3.]
 [ 1. -7.  1.  0.]
 [ 6. -1.  9.  8.]
 [ 3. -4.  9.  0.]]

Обратная матрица через LU:
[[ 0.1113470158 -0.1493276716  0.1325784383 -0.0417551309]
 [ 0.0113234253 -0.1677282378  0.0304317056 -0.0042462845]
 [-0.0320830385 -0.0247699929  0.0804435008  0.012031139

## Задание 1.2. Метод прогонки

### Условие

Реализовать метод прогонки в виде программы, задавая в качестве входных данных
ненулевые элементы матрицы системы и вектор правых частей.
Используя разработанное программное обеспечение, решить СЛАУ с трехдиагональной матрицей. 

### Требования лаборанта

При реализации запрещено хранить всю матрицу (т.к. она является разреженной). Хранить нужно только значения, лежащие на главной диагонали и двух побочных.

Необходимо в результате работы программы вывести:
1. Все прогоночные коэффициенты.
2. Итоговое решение системы уравнений.


### Входные данные

$\begin{cases} 13 \cdot x_1 - 5 \cdot x_2 = -66 \\ -4 \cdot x_1 + 9 \cdot x_2 - 5 \cdot x_3 = -47 \\ -x_2 - 12 \cdot x_3 - 6 \cdot x_4 = -43 \\ 6 \cdot x_3 + 20 \cdot x_4 - 5 \cdot x_5 = -74 \\ 4 \cdot x_4 + 5 \cdot x_5 = 14 \end{cases}$


In [5]:
data2 = {'lower': [-4, -1, 6, 4],
 'diagonal': [13, 9, -12, 20, 5],
 'upper': [-5, -5, -6, -5],
 'b': [-66, -47, -43, -74, 14]}


### Реализация

Прогонка: формулы (1.4)–(1.7), хранение только трёх диагоналей.


In [6]:
def thomas(lower, diagonal, upper, b):
    """Решить трёхдиагональную систему методом прогонки

    lower    — нижняя диагональ (n-1 элемент)
    diagonal — главная диагональ (n элементов)
    upper    — верхняя диагональ (n-1 элемент)
    b        — вектор правых частей (n элементов)
    P и Q    — прогоночные коэффициенты
    """
    n = len(diagonal)
    P = np.zeros(n)
    Q = np.zeros(n)
    x = np.zeros(n)

    # На первом шаге нет элемента нижней диагонали
    first_row_scale = max(abs(diagonal[0]), abs(upper[0]) if n > 1 else 0.0)
    if abs(diagonal[0]) <= DENOMINATOR_TOLERANCE * first_row_scale:
        raise ValueError("Нулевой знаменатель на первом шаге метода прогонки.")
    if n > 1:
        P[0] = -upper[0] / diagonal[0]
    Q[0] = b[0] / diagonal[0]

    # Прямой ход: вычисляем P_i и Q_i для следующих уравнений
    for i in range(1, n):
        denominator = diagonal[i] + lower[i - 1] * P[i - 1]
        row_scale = max(abs(diagonal[i]), abs(lower[i - 1]), abs(upper[i]) if i < n - 1 else 0.0)
        if abs(denominator) <= DENOMINATOR_TOLERANCE * row_scale:
            raise ValueError(f"Нулевой знаменатель на шаге {i + 1} метода прогонки.")
        if i < n - 1:
            P[i] = -upper[i] / denominator
        Q[i] = (b[i] - lower[i - 1] * Q[i - 1]) / denominator

    # У последнего уравнения нет элемента верхней диагонали
    P[-1] = 0.0

    # Обратный ход: используем x_i = P_i · x_(i+1) + Q_i
    x[-1] = Q[-1]
    for i in range(n - 2, -1, -1):
        x[i] = P[i] * x[i + 1] + Q[i]

    return P, Q, x


def main(data):
    lower = np.asarray(data["lower"], dtype=float)
    diagonal = np.asarray(data["diagonal"], dtype=float)
    upper = np.asarray(data["upper"], dtype=float)
    b = np.asarray(data["b"], dtype=float)
    P, Q, x = thomas(lower, diagonal, upper, b)
    print("Нижняя диагональ:", lower)
    print("Главная диагональ:", diagonal)
    print("Верхняя диагональ:", upper)
    print("Правая часть:", b)
    print_table(["i", "P_i", "Q_i", "x_i"], [(i + 1, P[i], Q[i], x[i]) for i in range(len(x))])
    # Для проверки не создаём плотную матрицу: используем только три диагонали
    residual = diagonal * x - b
    residual[1:] += lower * x[:-1]
    residual[:-1] += upper * x[1:]
    print(f"Максимальная невязка: {np.linalg.norm(residual, ord=np.inf):.3e}")
    return {"P": P, "Q": Q, "x": x, "residual": residual}

result2 = main(data2)


Нижняя диагональ: [-4. -1.  6.  4.]
Главная диагональ: [ 13.   9. -12.  20.   5.]
Верхняя диагональ: [-5. -5. -6. -5.]
Правая часть: [-66. -47. -43. -74.  14.]
i |             P_i |            Q_i | x_i
--+-----------------+----------------+----
1 |  0.384615384615 | -5.07692307692 |  -7
2 |  0.670103092784 |  -9.0206185567 |  -5
3 | -0.473555736371 |  4.10577705452 |   6
4 |  0.291397951442 | -5.74838770865 |  -4
5 |               0 |              6 |   6
Максимальная невязка: 1.421e-14


## Задание 1.3. Метод простых итераций и метод Зейделя

### Условие

Реализовать метод простых итераций и метод Зейделя в виде программ,
задавая в качестве входных данных матрицу системы, вектор правых частей
и точность вычислений. Используя разработанное программное обеспечение, решить СЛАУ.
Проанализировать количество итераций, необходимое для достижения заданной точности.

### Требования лаборанта

Необходимо реализовать оба метода и в результате работы программы вывести:
1. Заданную точность вычислений $ε$.
2. Количество итераций для каждого метода.
3. Точное или эталонное решение для проверки результата.
4. Сравнение методов по скорости сходимости.


### Входные данные

$\begin{cases} -23 \cdot x_1 - 7 \cdot x_2 + 5 \cdot x_3 + 2 \cdot x_4 = -26 \\ -7 \cdot x_1 - 21 \cdot x_2 + 4 \cdot x_3 + 9 \cdot x_4 = -55 \\ 9 \cdot x_1 + 5 \cdot x_2 - 31 \cdot x_3 - 8 \cdot x_4 = -58 \\ x_2 - 2 \cdot x_3 + 10 \cdot x_4 = -24 \end{cases}$


In [7]:
data3 = {'A': [[-23, -7, 5, 2], [-7, -21, 4, 9], [9, 5, -31, -8], [0, 1, -2, 10]],
 'b': [-26, -55, -58, -24],
 'epsilon': 1e-08,
 'max_iterations': 100000}


### Реализация

Простые итерации: (1.18)–(1.20); метод Зейделя и его оценка погрешности (§1.1.4).


In [8]:
def iteration_form(A, b):
    """Эквивалентная система x = alpha*x + beta, формула (1.18)"""
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    diagonal = np.diag(A)
    alpha = -A / diagonal[:, None]
    np.fill_diagonal(alpha, 0.0)
    beta = b / diagonal
    return alpha, beta


def jacobi(A, b, eps, max_iterations=100000):
    alpha, beta = iteration_form(A, b)
    q = float(np.linalg.norm(alpha, ord=np.inf))
    x = beta.copy()  # Начальное приближение из (1.19)
    history = []
    for iteration in range(1, max_iterations + 1):
        new_x = alpha @ x + beta
        step = float(np.linalg.norm(new_x - x, ord=np.inf))
        # При q >= 1 методичка допускает критерий по шагу без гарантии точности
        estimate = q / (1 - q) * step if q < 1 else step
        residual = float(np.linalg.norm(A @ new_x - b, ord=np.inf))
        history.append((iteration, new_x.copy(), step, estimate, residual))
        if estimate <= eps:
            return new_x, history
        x = new_x
    raise RuntimeError("Метод простых итераций не сошёлся за заданное число шагов.")


def seidel(A, b, eps, max_iterations=100000):
    alpha, beta = iteration_form(A, b)
    q = float(np.linalg.norm(alpha, ord=np.inf))
    c_norm = float(np.linalg.norm(np.triu(alpha, 1), ord=np.inf))
    x = beta.copy()
    history = []
    for iteration in range(1, max_iterations + 1):
        old_x = x.copy()
        for i in range(len(b)):
            # Слева используются уже обновлённые компоненты
            x[i] = beta[i] + alpha[i, :i] @ x[:i] + alpha[i, i + 1:] @ old_x[i + 1:]
        step = float(np.linalg.norm(x - old_x, ord=np.inf))
        estimate = c_norm / (1 - q) * step if q < 1 else step
        residual = float(np.linalg.norm(A @ x - b, ord=np.inf))
        history.append((iteration, x.copy(), step, estimate, residual))
        if estimate <= eps:
            return x, history
    raise RuntimeError("Метод Зейделя не сошёлся за заданное число шагов.")


def main(data):
    A = np.asarray(data["A"], dtype=float)
    b = np.asarray(data["b"], dtype=float)
    eps = data["epsilon"]
    alpha, beta = iteration_form(A, b)
    q = float(np.linalg.norm(alpha, ord=np.inf))
    c_norm = float(np.linalg.norm(np.triu(alpha, 1), ord=np.inf))
    print(f"Точность epsilon = {eps:g}")
    print_matrix("Исходная матрица A", A)
    print_matrix("Матрица alpha", alpha)
    print("beta =", beta)
    print(f"||alpha||_inf = {q:.12g}; ||C||_inf = {c_norm:.12g}")
    print("Начальное приближение x(0) = beta.")
    # Готовый решатель используется исключительно как эталон для проверки
    reference = np.linalg.solve(A, b)
    solutions = {}
    for name, method in (("Якоби", jacobi), ("Зейдель", seidel)):
        x, history = method(A, b, eps, data["max_iterations"])
        print(f"\nМетод {name}:")
        heading = "Оценка погрешности" if q < 1 else "Критерий по шагу"
        print_table(["k", "Норма шага", heading, "Невязка", "Отклонение от эталона"], [
            (k, step, estimate, residual, float(np.linalg.norm(value - reference, ord=np.inf)))
            for k, value, step, estimate, residual in history
        ])
        print("Решение:", x)
        solutions[name] = (x, history)
    print("\nЭталонное решение, только для проверки:", reference)
    print_table(["Метод", "Итераций", "Невязка", "Отклонение от эталона"], [
        (name, len(history), history[-1][4], float(np.linalg.norm(x - reference, ord=np.inf)))
        for name, (x, history) in solutions.items()
    ])
    counts = {name: len(history) for name, (_, history) in solutions.items()}
    minimum = min(counts.values())
    print("Меньшее число итераций: " + ", ".join(name for name, count in counts.items() if count == minimum))
    return {"solutions": solutions, "reference": reference, "q": q}


result3 = main(data3)


Точность epsilon = 1e-08

Исходная матрица A:
[[-23.  -7.   5.   2.]
 [ -7. -21.   4.   9.]
 [  9.   5. -31.  -8.]
 [  0.   1.  -2.  10.]]

Матрица alpha:
[[ 0.           -0.3043478261  0.2173913043  0.0869565217]
 [-0.3333333333  0.            0.1904761905  0.4285714286]
 [ 0.2903225806  0.1612903226  0.           -0.2580645161]
 [-0.           -0.1           0.2           0.          ]]
beta = [ 1.1304347826  2.619047619   1.8709677419 -2.4         ]
||alpha||_inf = 0.952380952381; ||C||_inf = 0.619047619048
Начальное приближение x(0) = beta.

Метод Якоби:
 k |        Норма шага | Оценка погрешности |           Невязка | Отклонение от эталона
---+-------------------+--------------------+-------------------+----------------------
 1 |     1.36997261738 |      27.3994523476 |     14.4174981634 |        0.468630201029
 2 |    0.626847746233 |      12.5369549247 |     5.15426325011 |        0.158217545205
 3 |    0.202782508906 |      4.05565017811 |     1.40988719449 |       0.044564963

## Задание 1.4. Метод вращений Якоби

### Условие

Реализовать метод вращений в виде программы, задавая в качестве входных данных матрицу
и точность вычислений. Используя разработанное программное обеспечение,
найти собственные значения и собственные векторы симметрических матриц.
Проанализировать зависимость погрешности вычислений от числа итераций. 

### Требования лаборанта

Необходимо предусмотреть и в результате работы программы вывести:
1. Заданную точность вычислений $ε$.
2. Найденные собственные значения.
3. Найденные собственные векторы и матрицу собственных векторов.
4. Проверку $A · V = V · Λ$, где $V$ — матрица собственных векторов, а $Λ$ — диагональная матрица собственных значений.


### Входные данные

$\begin{pmatrix} 5 & 5 & 3 \\ 5 & -4 & 1 \\ 3 & 1 & 2 \end{pmatrix}$


In [9]:
data4 = {'A': [[5, 5, 3], [5, -4, 1], [3, 1, 2]], 'epsilon': 1e-10, 'max_rotations': 100000}


### Реализация

Вращения Якоби и критерий по сумме квадратов внедиагональных элементов (§1.2.2).


In [10]:
def jacobi_eigen(A, eps, max_rotations=100000):
    """Собственные значения и векторы симметричной матрицы: §1.2.2"""
    work = np.asarray(A, dtype=float).copy()
    n = len(work)
    V = np.eye(n)
    upper = np.triu_indices(n, k=1)
    errors = []
    for step in range(max_rotations + 1):
        error = float(np.linalg.norm(work[upper]))
        errors.append(error)
        if error <= eps:
            order = np.argsort(np.diag(work))
            return np.diag(work)[order], V[:, order], errors
        if step == max_rotations:
            break
        # Выбор элемента — вспомогательный argmax; само вращение строится явно
        position = int(np.argmax(np.abs(work[upper])))
        p, q = upper[0][position], upper[1][position]
        tau = (work[q, q] - work[p, p]) / (2 * work[p, q])
        t = np.sign(tau) / (abs(tau) + np.hypot(1.0, tau)) if tau != 0 else 1.0
        c = 1 / np.sqrt(1 + t * t)
        s = t * c
        rotation = np.eye(n)
        rotation[p, p] = rotation[q, q] = c
        rotation[p, q], rotation[q, p] = s, -s
        work = rotation.T @ work @ rotation
        work[p, q] = work[q, p] = 0.0  # Остаток округления в обнуляемой паре
        V = V @ rotation
    raise RuntimeError("Метод вращений не сошёлся за заданное число шагов.")


def main(data):
    A = np.asarray(data["A"], dtype=float)
    eps = data["epsilon"]
    values, V, errors = jacobi_eigen(A, eps, data["max_rotations"])
    Lambda = np.diag(values)
    print(f"Точность epsilon = {eps:g}; вращений: {len(errors) - 1}")
    print("Собственные значения:", values)
    for name, matrix in (("Исходная матрица A", A), ("Собственные векторы по столбцам V", V),
                         ("Диагональная матрица Lambda", Lambda), ("A @ V", A @ V), ("V @ Lambda", V @ Lambda)):
        print_matrix(name, matrix)
    print_table(["Проверка", "Максимальное расхождение"], [
        ("A @ V = V @ Lambda", max_abs_difference(A @ V, V @ Lambda)),
        ("V.T @ V = E", max_abs_difference(V.T @ V, np.eye(len(A)))),
    ])
    print("\nКритерий остановки: t(A) = норма верхнего внедиагонального треугольника.")
    print_table(["k", "t(A_k)"], list(enumerate(errors)))
    return {"values": values, "V": V, "history": errors}

result4 = main(data4)


Точность epsilon = 1e-10; вращений: 7
Собственные значения: [-6.2393735233  0.5339061455  8.7054673777]

Исходная матрица A:
[[ 5.  5.  3.]
 [ 5. -4.  1.]
 [ 3.  1.  2.]]

Собственные векторы по столбцам V:
[[-0.415207741  -0.3717040284  0.8303244228]
 [ 0.908807712  -0.2104910556  0.3602250103]
 [ 0.0408787768  0.9041734517  0.4252050033]]

Диагональная матрица Lambda:
[[-6.2393735233  0.            0.          ]
 [ 0.            0.5339061455  0.          ]
 [ 0.            0.            8.7054673777]]

A @ V:
[[ 2.5906361856 -0.1984550651  7.2283621755]
 [-5.6703907762 -0.1123824682  3.135927076 ]
 [-0.2550579573  0.4827437625  3.7016082854]]

V @ Lambda:
[[ 2.5906361856 -0.1984550651  7.2283621755]
 [-5.6703907762 -0.1123824682  3.135927076 ]
 [-0.2550579573  0.4827437625  3.7016082854]]
          Проверка | Максимальное расхождение
-------------------+-------------------------
A @ V = V @ Lambda |        1.90514271026e-13
       V.T @ V = E |          8.881784197e-16

Критерий оста

## Задание 1.5. QR-разложение

### Условие

Реализовать алгоритм $QR$–разложения матриц в виде программы.
На его основе разработать программу, реализующую $QR$–алгоритм решения
полной проблемы собственных значений произвольных матриц,
задавая в качестве входных данных матрицу и точность вычислений.
С использованием разработанного программного обеспечения найти собственные значения матрицы.

### Требования лаборанта

Необходимо в результате работы программы вывести:
1. Исходную матрицу $A$.
2. Матрицы $Q$ и $R$.
3. Результат умножения $Q · R$.
4. Найденные собственные значения.

Также необходимо привести собственный пример матрицы с комплексно-сопряжёнными собственными значениями.

Только в этой части допускается использование **NumPy** для нахождения и проверки комплексно-сопряжённой пары собственных значений.


### Входные данные

$\begin{pmatrix} 5 & -5 & -6 \\ -1 & -8 & -5 \\ 2 & 7 & -3 \end{pmatrix}$

Собственный пример: $\begin{pmatrix}0 & -1 \\ 1 & 0\end{pmatrix}$.


In [11]:
data5 = {'A': [[5, -5, -6], [-1, -8, -5], [2, 7, -3]],
 'epsilon': 1e-10,
 'max_iterations': 20000,
 'example': [[0, -1], [1, 0]]}


### Реализация

Отражения Хаусхолдера (1.29), (1.30) и базовый QR-процесс с контролем блоков (§1.2.4).


In [12]:
def qr(A):
    """Выполнить QR-разложение A = Q · R отражениями Хаусхолдера

    Q — ортогональная матрица; R — верхняя треугольная матрица
    """
    n = len(A)
    R = A.copy().astype(float)
    Q = np.eye(n)

    for k in range(n - 1):
        x = R[k:, k].copy()  # Часть столбца, которую нужно обнулить ниже диагонали
        norm = np.linalg.norm(x)
        if norm == 0:
            continue

        # Строим вектор отражения. Знак выбран так, чтобы избежать потери точности
        sign = 1 if x[0] >= 0 else -1
        x[0] += sign * norm
        x /= np.linalg.norm(x)

        # Применяем отражение к R слева, а соответствующее преобразование накапливаем в Q
        R[k:, k:] -= 2 * np.outer(x, x @ R[k:, k:])
        Q[:, k:] -= 2 * np.outer(Q[:, k:] @ x, x)

    # После отражений Хаусхолдера R должна быть верхнетреугольной
    # Обнуляем только нижнюю часть, где остались погрешности округления
    R[np.tril_indices(n, k=-1)] = 0
    return Q, R


def eig2(a, b, c, d):
    """Найти собственные значения матрицы 2×2, включая комплексную пару при необходимости"""
    # Для матрицы [[a, b], [c, d]] используем формулу через след и дискриминант
    discriminant = (a - d) ** 2 + 4 * b * c
    half_trace = (a + d) / 2
    if discriminant >= 0:
        root = np.sqrt(discriminant) / 2
        return [float(half_trace + root), float(half_trace - root)]

    # Отрицательный дискриминант означает два комплексно-сопряжённых значения
    root = np.sqrt(-discriminant) / 2
    return [complex(half_trace, root), complex(half_trace, -root)]


def qr_blocks(H, eps, previous_pairs):
    """Выделить блоки 1x1 и 2x2 и проверить критерии §1.2.4"""
    n = len(H)
    pairs = {i: np.asarray(eig2(H[i, i], H[i, i + 1], H[i + 1, i], H[i + 1, i + 1]), dtype=complex)
             for i in range(n - 1)}
    values = []
    tail_max = 0.0
    change_max = 0.0
    i = 0
    while i < n:
        tail_one = float(np.linalg.norm(H[i + 1:, i]))
        if tail_one <= eps:
            values.append(float(H[i, i]))
            tail_max = max(tail_max, tail_one)
            i += 1
            continue
        # Для блока проверяется вся часть обоих столбцов ниже блока
        tail_two = float(np.linalg.norm(H[i + 2:, i:i + 2]))
        tail_max = max(tail_max, tail_two)
        if i in previous_pairs:
            current, old = pairs[i], previous_pairs[i]
            change = min(float(np.max(np.abs(current - old))),
                         float(np.max(np.abs(current - old[::-1]))))
        else:
            change = math.inf
        change_max = max(change_max, change)
        if tail_two > eps or change > eps:
            return None, pairs, tail_max, change_max
        values.extend(pairs[i].tolist())
        i += 2
    return values, pairs, tail_max, change_max


def qr_eigenvalues(A, eps, max_iterations=20000):
    """Базовый QR-процесс A_k = Q_k R_k, A_(k+1) = R_k Q_k из §1.2.4"""
    H = np.asarray(A, dtype=float).copy()
    previous_pairs = {}
    history = []
    for iteration in range(max_iterations + 1):
        values, pairs, tail, change = qr_blocks(H, eps, previous_pairs)
        history.append((iteration, tail, change))
        if values is not None:
            return np.real_if_close(np.asarray(values)), H, history
        previous_pairs = pairs
        if iteration < max_iterations:
            Q, R = qr(H)
            H = R @ Q
    raise RuntimeError("QR-процесс не сошёлся за заданное число итераций.")


def main(data):
    A = np.asarray(data["A"], dtype=float)
    eps = data["epsilon"]
    Q, R = qr(A)
    print_matrix("Исходная матрица A", A)
    print_matrix("Матрица Q", Q)
    print_matrix("Матрица R", R)
    print_matrix("Произведение Q @ R", Q @ R)
    print(f"Максимальное расхождение A и Q @ R: {max_abs_difference(A, Q @ R):.3e}")
    print(f"Проверка Q.T @ Q = E: {max_abs_difference(Q.T @ Q, np.eye(len(A))):.3e}")
    values, H, history = qr_eigenvalues(A, eps, data["max_iterations"])
    print(f"\nТочность epsilon = {eps:g}; QR-итераций: {len(history) - 1}")
    print_table(["k", "Остаток выделения блоков", "Изменение корней блоков 2x2"],
                [(k, tail, change if math.isfinite(change) else "—") for k, tail, change in history])
    print_matrix("Итоговая матрица QR-процесса", H)
    print("Собственные значения собственного QR-алгоритма:", values)

    # NumPy применён только после собственного алгоритма для проверки спектра
    reference = np.linalg.eigvals(A)
    print("Проверка спектра средствами NumPy:", reference)
    print("\nСобственный пример с комплексно-сопряжённой парой:")
    example = np.asarray(data["example"], dtype=float)
    print_matrix("Матрица примера", example)
    example_values, _, example_history = qr_eigenvalues(example, eps, data["max_iterations"])
    print("Собственный QR-алгоритм:", example_values)
    print("Проверка NumPy:", np.linalg.eigvals(example))
    return {"Q": Q, "R": R, "values": values, "H": H, "history": history,
            "reference": reference, "example_values": example_values}


result5 = main(data5)



Исходная матрица A:
[[ 5. -5. -6.]
 [-1. -8. -5.]
 [ 2.  7. -3.]]

Матрица Q:
[[-0.9128709292 -0.3834824944 -0.1400280084]
 [ 0.1825741858 -0.69026849    0.700140042 ]
 [-0.3651483717  0.6135719911  0.700140042 ]]

Матрица R:
[[-5.4772255751  0.5477225575  5.6597997609]
 [ 0.           11.7345643294  3.9115214431]
 [ 0.            0.           -4.7609522857]]

Произведение Q @ R:
[[ 5. -5. -6.]
 [-1. -8. -5.]
 [ 2.  7. -3.]]
Максимальное расхождение A и Q @ R: 2.665e-15
Проверка Q.T @ Q = E: 2.220e-16

Точность epsilon = 1e-10; QR-итераций: 56
 k | Остаток выделения блоков | Изменение корней блоков 2x2
---+--------------------------+----------------------------
 0 |            7.28010988928 |                           —
 1 |             3.3993463424 |               2.26792318801
 2 |            7.26651068643 |               2.47285425977
 3 |            6.18481807212 |               8.16587260791
 4 |             4.3998547839 |               7.83684375499
 5 |            9.15572787912